# Tire Shop RAG Assistant — пайплайн

RAG-ассистент для магазина шин: HTML-заглушки → Markdown → эмбеддинги → ChromaDB → rerank → локальная LLM (Saiga Mistral 7B GGUF через llama.cpp).

> ⚠️ Все данные в репозитории — **фиктивные**. Парсеры работают только на локальных HTML-заглушках из `data/samples/`.

Это **второй интерфейс** запуска пайплайна (первый — `python main.py` локально в VS Code).Ноутбук работает и в **Google Colab** (рекомендуется GPU: *Среда выполнения → Сменить среду выполнения → T4 GPU*), и локально в **VS Code / Jupyter**.

---


In [ ]:
import os
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    GITHUB_USER = "Veskernout"   # ← замени на свой GitHub-аккаунт
    REPO_NAME = "tire-shop-rag-assistant"
    repo_dir = Path("/content") / REPO_NAME
    if not repo_dir.exists():
        import subprocess

        subprocess.run(
            ["git", "clone", f"https://github.com/{GITHUB_USER}/{REPO_NAME}.git", str(repo_dir)],
            check=True,
        )
    os.chdir(repo_dir)

REPO_ROOT = Path.cwd()
sys.path.insert(0, str(REPO_ROOT))
print(f"REPO_ROOT: {REPO_ROOT}")
print(f"IN_COLAB:  {IN_COLAB}")


In [ ]:
# Первый запуск: установит зависимости (в Colab начнётся установкой пинов из requirements.txt).
# --no-deps не трогает предустановленные в Colab пакеты (numpy, opencv и т.д.),
# чтобы избежать бинарных конфликтов (numpy.dtype size changed).
try:
    import llama_index  # noqa: F401
except ImportError:
    %pip install -q -r requirements.txt --no-deps

import sys
print(f"Python version: {sys.version}")
if sys.version_info >= (3, 12):
    print("⚠️  Проект тестировался на Python 3.11. На 3.13 возможны нюансы с бинарными пакетами.")

for _mod in ["numpy", "llama_index", "chromadb", "sentence_transformers", "llama_cpp", "huggingface_hub"]:
    try:
        __import__(_mod)
    except ImportError as _e:
        print(f"WARNING: {_mod} not available: {_e}")


In [ ]:
import shutil
import subprocess


def _has_nvidia_gpu() -> bool:
    nvidia_smi = shutil.which("nvidia-smi")
    if not nvidia_smi:
        return False
    result = subprocess.run([nvidia_smi, "-L"], capture_output=True, text=True)
    return "GPU" in result.stdout


HAS_GPU = _has_nvidia_gpu()
print("GPU active:", HAS_GPU)

# llama-cpp-python ставится из PyPI (CPU-сборка). Если в среде доступен GPU-wheel,
# LlamaCPP с n_gpu_layers=-1 сам загрузит слои на видеокарту. Длительная пересборка
# из исходников под CUDA в Colab отключена, чтобы ноутбук запускался быстро и стабильно.


In [ ]:
from src.pipeline import (
    build_knowledge_base,
    build_rag_stack,
    detect_gpu_layers,
    run_evaluation,
)

# Парсинг → эмбеддинги → индекс ChromaDB → скачивание GGUF → LLM + reranker.
build_knowledge_base()
gpu_layers = detect_gpu_layers()
print(f"n_gpu_layers={gpu_layers} ({'GPU' if gpu_layers == -1 else 'CPU'})")

engine = build_rag_stack(gpu_layers=gpu_layers)
print("Query engine готов.")


In [ ]:
# Прогон QA-пар из evaluation/qa_pairs.json + сводная статистика.
stats = run_evaluation(engine)

# stats: {"total": ..., "passed": ..., "accuracy": ...}


In [ ]:
print("Задай вопрос ассистенту (пустая строка — выход):")
while True:
    question = input("Вы: ")
    if not question.strip():
        break
    print("Ассистент:", engine.query(question))
    print()


## Заметки

- **Первый запуск** скачает модели: ~470 МБ (эмбеддинги MiniLM-L12-v2), ~460 МБ (cross-encoder для реранка) и ~4.2 ГБ (Saiga Mistral 7B Q4_K_M).
- **Без GPU** всё тоже работает, но медленно. В Colab включите GPU-рантайм до перезапуска сессии.
- **Локально** в VS Code достаточно `python main.py` — пайплайн тот же, в ноутбуке он разбит на ячейки.
- Другую GGUF-модель можно подключить переменными окружения `RAG_MODEL_REPO` / `RAG_MODEL_FILE` перед сборкой стека (пример: `Qwen/Qwen2.5-0.5B-Instruct-GGUF`).
